# 🎬 Movie Recommendation System
This project builds a simple  content-based movie recommender using the TMDB 5000 Movies dataset**.  
It recommends similar movies based on a film’s *overview* and *genres*, using TF-IDF vectorization and cosine similarity.

---

### 🧠 Objective
The goal is to help users find movies similar to their favorites, purely based on the content description — no user ratings needed.

---

### 🧰 Tools Used
- Python  
- Pandas, NumPy  
- Scikit-learn  
- Google Colab  

---

### 📁 Dataset
Dataset: [TMDB 5000 Movies Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)


In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

pd.set_option('display.max_colwidth', 200)
print("✅ Libraries imported successfully")


✅ Libraries imported successfully


In [2]:
# Load dataset
file_path = "/content/tmdb_5000_movies.csv"
df = pd.read_csv(file_path)

# Check shape and first few rows
print("Shape of dataset:", df.shape)
df.head(3)


Shape of dataset: (4803, 20)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""Science Fiction""}]",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""space war""}, {""id"": 3388, ""name"": ""space colony""}, {""id"": 3679, ""name"": ""society""}, {""id"": 3801, ""name...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century Fox Film Corporation"", ""id"": 306}, {""name"": ""Dune Entertainment"", ""id"": 444}, {""name"": ""Lightstorm Entertainment"", ""id""...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}]",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic island""}, {""id"": 1319, ""name"": ""east india trading company""}, {""id"": 2038, ""name"": ""love of one's life...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Films"", ""id"": 130}, {""name"": ""Second Mate Productions"", ""id"": 19936}]","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret agent""}, {""id"": 9663, ""name"": ""sequel""}, {""id"": 14555, ""name"": ""mi6""}, {""id"": 156095, ""name"": ""brit...",en,Spectre,"A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit ...",107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""name"": ""Danjaq"", ""id"": 10761}, {""name"": ""B24"", ""id"": 69434}]","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""}, {""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}, {""iso_639_1"": ""it"", ""name"": ""Italiano""}, {""iso_639_1"": ""de"", ""na...",Released,A Plan No One Escapes,Spectre,6.3,4466


> 💡 **Insight:**  
> The dataset contains around 4800 movies with detailed metadata such as titles, genres, overviews, and other information.  
> We'll use only the `title`, `genres`, and `overview` columns to build our recommendation model.


In [5]:
df = df[['id', 'title', 'overview', 'genres']]
df.head(3)


,id,title,overview,genres
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""Science Fiction""}]"
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]"
2,206647,Spectre,"A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit ...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]"


> 🧩 **Data Cleaning Notes:**  
> - Extracted genre names from the list of dictionaries (e.g., `[{ "id": 28, "name": "Action" }]` → `"Action"`)  
> - Combined `overview` and `genres` into a single text field called `content`  
> - Filled missing overviews with blank strings for stability


In [6]:
# Convert 'genres' from stringified list to plain text
def parse_genres(genres_str):
    try:
        genres_list = ast.literal_eval(genres_str)
        return " ".join([g['name'] for g in genres_list])
    except:
        return ""

df['genres'] = df['genres'].apply(parse_genres)

# Replace missing overviews
df['overview'] = df['overview'].fillna("")

# Combine text data (overview + genres)
df['content'] = df['overview'] + " " + df['genres']

print("✅ Cleaned and prepared text data")
df.head(3)


✅ Cleaned and prepared text data


,id,title,overview,genres,content
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.",Action Adventure Fantasy Science Fiction,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fan..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.",Adventure Fantasy Action,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems. Adventure Fantasy A..."
2,206647,Spectre,"A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit ...",Action Adventure Crime,"A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit ..."


In [7]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (4803, 20978)


> ⚙️ **Model Explanation:**  
> - Used TF-IDF (Term Frequency–Inverse Document Frequency) to convert text into numerical form.  
> - Used cosine similarity to measure how close movies are to one another based on their textual content.  
> - This means two movies are considered similar if their overviews and genres use similar words.


In [8]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Reset index for easy lookup
df = df.reset_index()
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

print("✅ Cosine similarity matrix ready")


✅ Cosine similarity matrix ready


In [9]:
def recommend(title, n=10):
    if title not in indices:
        return f"❌ '{title}' not found in dataset."

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # skip the movie itself
    movie_indices = [i[0] for i in sim_scores]
    return df[['title', 'genres', 'overview']].iloc[movie_indices]


In [10]:
recommend("Avatar")


,title,genres,overview
3604,Apollo 18,Horror Thriller Science Fiction,"Officially, Apollo 17 was the last manned mission to the moon. But a year later in 1973, three American astronauts were sent on a secret mission to the moon funded by the US Department of Defense...."
4401,The Helix... Loaded,Action Comedy Science Fiction,
634,The Matrix,Action Science Fiction,"Set in the 22nd century, The Matrix tells the story of a computer hacker who joins a group of underground insurgents fighting the vast and powerful computers who now rule the earth."
2130,The American,Crime Drama Thriller,"Dispatched to a small Italian town to await further orders, assassin Jack embarks on a double life that may be more relaxing than is good for him."
1341,The Inhabited Island,Action Fantasy Science Fiction Thriller,"On the threshold of 22nd century, furrowing the space, protagonist from the Free Search Group makes emergency landing on an unknown planet where he must stay. People who are living on this planet ..."
529,Tears of the Sun,Action Drama War,"Navy SEAL Lieutenant A.K. Waters and his elite squadron of tactical specialists are forced to choose between their duty and their humanity, between following orders by ignoring the conflict that s..."
311,The Adventures of Pluto Nash,Action Comedy Science Fiction,"The year is 2087, the setting is the moon. Pluto Nash, the high-flying successful owner of the hottest nightclub in the universe, finds himself in trouble when he refuses to sell his club to lunar..."
942,The Book of Life,Romance Animation Adventure Comedy Family Fantasy,"The journey of Manolo, a young man who is torn between fulfilling the expectations of his family and following his heart. Before choosing which path to follow, he embarks on an incredible adventur..."
1610,Hanna,Action Thriller Adventure,"A 16-year-old girl raised by her father to be the perfect assassin is dispatched on a mission across Europe. Tracked by a ruthless operatives, she faces startling revelations about her existence a..."
2628,Blood and Chocolate,Drama Fantasy Horror Romance,A young teenage werewolf is torn between honoring her family's secret and her love for a man.


> 💡 **Example Recommendation:**  
> When we search for movies similar to *Avatar*, the system recommends other science-fiction and adventure titles such as *Guardians of the Galaxy* and *Star Trek*.  
> This shows the model successfully captures the underlying theme of the movie.


## 🧾 Summary

This project demonstrates a simple but effective content-based recommendation system.  
It identifies similar movies based on descriptions and genres — no user ratings required.

### ✅ Key Takeaways
- TF-IDF and cosine similarity can be powerful for text-based recommendations.  
- Simple NLP techniques can produce impressive results for movie discovery systems.  
- The model can be extended with user data or additional metadata (e.g., keywords, cast).

### 🚀 Next Steps
- Add user ratings and collaborative filtering to improve recommendations.  
- Combine with deep learning embeddings for smarter matching.  
- Deploy as a web app using Streamlit.


